In [217]:
from pathlib import Path
import sys


sys.path.append(str(Path().resolve().parent))


                           

                                                            
                                            

BASE_DIR = Path().resolve()
PROJECT_ROOT = BASE_DIR.parent   #to go un up the the root
DATA_DIR = PROJECT_ROOT / "data"

# Root directory of dataset
root = Path(DATA_DIR)

In [218]:
import pandas as pd

def build_df(root):

    label_map = {
        1: "correct",
        2: "fast",
        3: "low_amplitude"
    }

    dfs = []   # store all partial dataframes

    for subject_dir in root.glob("s*"):
        for exercise_dir in subject_dir.glob("e*"):
            for unit_dir in exercise_dir.glob("u*"):

                file_path = unit_dir / "test.txt"

                df = pd.read_csv(file_path, sep=";")

                df["subject"] = subject_dir.name
                df["exercise"] = exercise_dir.name
                df["unit"] = unit_dir.name

                

                label_map = {
                    0: "correct",
                    1: "fast",
                    2: "low_amplitude"
                }

                segment_length = len(df) // 30

                df["execution_type"] = None

                for i in range(30):
                    start = i * segment_length
                    end = (i + 1) * segment_length

                    label_group = i // 10
                    label = label_map[label_group]

                    df.loc[start:end, "execution_type"] = label

                dfs.append(df)   # collect dataframe

    final_df = pd.concat(dfs, ignore_index=True)

    return final_df

In [219]:
df=build_df(root)


In [220]:
df[(df['subject']=='s1') & (df['exercise']=='e1') ]

,time index,acc_x,acc_y,acc_z,gyr_x,gyr_y,gyr_z,mag_x,mag_y,mag_z,subject,exercise,unit,execution_type
0,1,-9.685645,-1.645149,0.505022,-0.020696,0.009202,-0.008566,0.589728,0.453403,-0.075234,s1,e1,u1,correct
1,2,-9.648184,-1.645353,0.513125,-0.008165,-0.001407,-0.003256,0.587024,0.453644,-0.075593,s1,e1,u1,correct
2,3,-9.700570,-1.615223,0.512321,-0.004447,0.011059,-0.008589,0.589691,0.454598,-0.075525,s1,e1,u1,correct
3,4,-9.685627,-1.630183,0.497591,-0.026110,0.009183,-0.008554,0.589240,0.452864,-0.074705,s1,e1,u1,correct
4,5,-9.655697,-1.630194,0.460742,-0.008109,0.001231,-0.005950,0.589647,0.452882,-0.076429,s1,e1,u1,correct
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29740,5945,-4.223868,-7.541800,-4.607269,-0.016858,-0.001904,-0.011902,0.003246,0.725318,0.231512,s1,e1,u5,None
29741,5946,-4.253123,-7.531929,-4.612775,-0.006980,-0.021866,-0.013705,0.005410,0.724710,0.231259,s1,e1,u5,None
29742,5947,-4.233624,-7.536894,-4.607502,-0.013551,-0.001552,0.000465,0.004094,0.725666,0.233084,s1,e1,u5,None
29743,5948,-4.228771,-7.517463,-4.622438,-0.020245,-0.013049,-0.019876,0.004089,0.724876,0.232979,s1,e1,u5,None


In [222]:
merged_df = df.pivot(
    index=["subject", "exercise", "time index"],
    columns="unit",
    values=["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z", "mag_x", "mag_y", "mag_z"]
)

merged_df.columns = [f"{sensor}_{unit}" for sensor, unit in merged_df.columns]
merged_df = merged_df.reset_index()




In [224]:
merged_df[(merged_df['subject']=='s1') & (merged_df['exercise']=='e1') ]

,subject,exercise,time index,acc_x_u1,acc_x_u2,acc_x_u3,acc_x_u4,acc_x_u5,acc_y_u1,acc_y_u2,...,mag_y_u1,mag_y_u2,mag_y_u3,mag_y_u4,mag_y_u5,mag_z_u1,mag_z_u2,mag_z_u3,mag_z_u4,mag_z_u5
0,s1,e1,1,-9.685645,-9.268303,-3.490739,9.018139,-4.191311,-1.645149,2.981149,...,0.453403,-0.586875,-0.803904,-0.000218,0.725706,-0.075234,-0.038496,-0.069200,-0.574391,0.232402
1,s1,e1,2,-9.648184,-9.260987,-3.445827,9.044963,-4.188912,-1.645353,2.958806,...,0.453644,-0.588163,-0.803221,-0.000222,0.724973,-0.075593,-0.038908,-0.070015,-0.573762,0.232350
2,s1,e1,3,-9.700570,-9.260972,-3.475073,9.077281,-4.181551,-1.615223,2.966184,...,0.454598,-0.586852,-0.803683,-0.001231,0.725197,-0.075525,-0.038825,-0.069519,-0.575043,0.233151
3,s1,e1,4,-9.685627,-9.260754,-3.474895,9.032722,-4.181585,-1.630183,3.010663,...,0.452864,-0.587912,-0.802428,-0.001238,0.726052,-0.074705,-0.038753,-0.069173,-0.574773,0.231525
4,s1,e1,5,-9.655697,-9.260783,-3.467499,8.993878,-4.186467,-1.630194,2.995905,...,0.452882,-0.587155,-0.802398,0.000118,0.724997,-0.076429,-0.038180,-0.069616,-0.576355,0.231562
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5944,s1,e1,5945,-9.665146,-9.376658,-3.468145,8.951390,-4.223868,-1.714249,2.548301,...,0.452371,-0.565074,-0.802334,-0.070486,0.725318,-0.092053,-0.040609,-0.079843,-0.589032,0.231512
5945,s1,e1,5946,-9.650109,-9.399016,-3.468054,8.936646,-4.253123,-1.661896,2.548481,...,0.454637,-0.565617,-0.802279,-0.068400,0.724710,-0.094303,-0.040415,-0.077974,-0.588567,0.231259
5946,s1,e1,5947,-9.650065,-9.428883,-3.497979,8.945643,-4.233624,-1.631988,2.548632,...,0.453709,-0.564694,-0.802558,-0.070792,0.725666,-0.094820,-0.040438,-0.078147,-0.587765,0.233084
5947,s1,e1,5948,-9.650072,-9.376829,-3.482882,9.073227,-4.228771,-1.631962,2.518608,...,0.453846,-0.565730,-0.802543,-0.070244,0.724876,-0.093984,-0.040850,-0.077960,-0.588575,0.232979
